In [15]:
# ============================================================
# MyGPT2 Validation Evaluation
# Cell 1 - Imports
# ============================================================

from pathlib import Path
import sys
import math
import time
import json

import torch
import torch.nn.functional as F

print("=" * 75)
print("MyGPT2 - Validation Evaluation")
print("=" * 75)

print("PyTorch Version :", torch.__version__)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("CUDA Available  : YES")
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("CUDA Available  : NO")

print("Device          :", DEVICE)

print("=" * 75)

MyGPT2 - Validation Evaluation
PyTorch Version : 2.13.0+cu132
CUDA Available  : YES
GPU             : NVIDIA GeForce RTX 5060 Ti
Device          : cuda


In [16]:
# ============================================================
# Cell 2 - Project Paths
# ============================================================

PROJECT_ROOT = Path(r"D:\Gpt2_v01").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "checkpoints"
    / "final_step_00010000.pt"
)

TOKENIZER_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tokenizer"
    / "tokenizer.json"
)

print("=" * 75)
print("Project Paths")
print("=" * 75)

print("Project Root :", PROJECT_ROOT)
print("Checkpoint   :", CHECKPOINT_PATH)
print("Tokenizer    :", TOKENIZER_PATH)

print()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist:\n{PROJECT_ROOT}"
    )

if not CHECKPOINT_PATH.exists():
    checkpoint_dir = (
        PROJECT_ROOT
        / "artifacts"
        / "checkpoints"
    )

    print("Available checkpoints:")
    
    if checkpoint_dir.exists():
        for file in sorted(checkpoint_dir.glob("*.pt")):
            print("  -", file.name)

    raise FileNotFoundError(
        f"\nCheckpoint not found:\n{CHECKPOINT_PATH}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found:\n{TOKENIZER_PATH}"
    )

print("Project Root : ✅ FOUND")
print("Checkpoint   : ✅ FOUND")
print("Tokenizer    : ✅ FOUND")

print("=" * 75)

Project Paths
Project Root : D:\Gpt2_v01
Checkpoint   : D:\Gpt2_v01\artifacts\checkpoints\final_step_00010000.pt
Tokenizer    : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json

Project Root : ✅ FOUND
Checkpoint   : ✅ FOUND
Tokenizer    : ✅ FOUND


In [17]:
print("Checkpoint exists :", CHECKPOINT_PATH.exists())
print("Tokenizer exists  :", TOKENIZER_PATH.exists())

print()
print("Checkpoint size   :",
      f"{CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB")

print("Tokenizer size    :",
      f"{TOKENIZER_PATH.stat().st_size / (1024**2):.2f} MB")

Checkpoint exists : True
Tokenizer exists  : True

Checkpoint size   : 1259.35 MB
Tokenizer size    : 2.16 MB


In [18]:
# ============================================================
# Cell 3 - MyGPT2 Imports
# ============================================================

from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer
from training.checkpoint import load_checkpoint

print("GPTConfig       : ✅")
print("MyGPTModel      : ✅")
print("MyGPTTokenizer  : ✅")
print("load_checkpoint : ✅")

print()
print("All MyGPT2 imports successful.")

GPTConfig       : ✅
MyGPTModel      : ✅
MyGPTTokenizer  : ✅
load_checkpoint : ✅

All MyGPT2 imports successful.


In [20]:
from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer

from training.checkpoint import load_checkpoint

# IMPORTANT:
# Replace this import with the exact Dataset class used by train.py.
#
# Example:
# from training.dataset import TinyStoriesDataset
#
# If your train.py imports the dataset from another module,
# use that exact import here.

In [22]:
# ============================================================
# Cell 4 - Model Configuration
# ============================================================

config = GPTConfig()


def get_config_value(config, names, default=None):
    """
    Safely retrieve the first available configuration attribute.
    """
    for name in names:
        if hasattr(config, name):
            value = getattr(config, name)

            if value is not None:
                return value

    return default


# ------------------------------------------------------------
# Vocabulary
# ------------------------------------------------------------

VOCAB_SIZE = get_config_value(
    config,
    [
        "vocab_size",
    ],
)


# ------------------------------------------------------------
# Sequence Length
# ------------------------------------------------------------
#
# Your training run used:
#
# Sequence Length : 512
#
# We first try to find the value in GPTConfig.
# If GPTConfig does not expose it, use the exact
# sequence length used during training.
# ------------------------------------------------------------

SEQUENCE_LENGTH = get_config_value(
    config,
    [
        "context_length",
        "block_size",
        "max_seq_len",
        "max_sequence_length",
        "sequence_length",
        "seq_length",
        "context_size",
        "n_positions",
        "max_position_embeddings",
        "max_context_length",
    ],
    default=512,
)


# ------------------------------------------------------------
# Hidden Size
# ------------------------------------------------------------

HIDDEN_SIZE = get_config_value(
    config,
    [
        "hidden_size",
        "n_embd",
        "embedding_dim",
        "d_model",
    ],
)


# ------------------------------------------------------------
# Transformer Layers
# ------------------------------------------------------------

NUM_LAYERS = get_config_value(
    config,
    [
        "num_layers",
        "n_layer",
        "layers",
        "num_hidden_layers",
    ],
)


# ------------------------------------------------------------
# Attention Heads
# ------------------------------------------------------------

NUM_HEADS = get_config_value(
    config,
    [
        "num_heads",
        "n_head",
        "attention_heads",
        "num_attention_heads",
    ],
)


# ------------------------------------------------------------
# Intermediate / FFN Size
# ------------------------------------------------------------

INTERMEDIATE_SIZE = get_config_value(
    config,
    [
        "intermediate_size",
        "ffn_size",
        "hidden_dim",
    ],
)


# ============================================================
# Display Configuration
# ============================================================

print("=" * 75)
print("Model Configuration")
print("=" * 75)

print(
    f"Vocabulary Size      : "
    f"{VOCAB_SIZE:,}"
)

print(
    f"Sequence Length      : "
    f"{SEQUENCE_LENGTH}"
)

print(
    f"Hidden Size          : "
    f"{HIDDEN_SIZE}"
)

print(
    f"Transformer Layers   : "
    f"{NUM_LAYERS}"
)

print(
    f"Attention Heads      : "
    f"{NUM_HEADS}"
)

print(
    f"Intermediate Size    : "
    f"{INTERMEDIATE_SIZE}"
)

print("=" * 75)


# ============================================================
# Validation
# ============================================================

if VOCAB_SIZE is None:
    raise RuntimeError(
        "Could not determine vocabulary size "
        "from GPTConfig."
    )


if SEQUENCE_LENGTH is None:
    raise RuntimeError(
        "Could not determine sequence length."
    )


if SEQUENCE_LENGTH != 512:
    print(
        "WARNING: The detected sequence length is "
        f"{SEQUENCE_LENGTH}, but the model was trained "
        "with sequence length 512."
    )
else:
    print(
        "Sequence Length : ✅ 512 "
        "(matches training configuration)"
    )


# ============================================================
# Expected Configuration Check
# ============================================================

EXPECTED_CONFIG = {
    "vocab_size": 32000,
    "sequence_length": 512,
    "hidden_size": 768,
    "num_layers": 12,
    "num_heads": 12,
    "intermediate_size": 3072,
}


print()
print("=" * 75)
print("Expected Training Configuration Check")
print("=" * 75)

checks = {
    "Vocabulary Size": (
        VOCAB_SIZE,
        EXPECTED_CONFIG["vocab_size"],
    ),
    "Sequence Length": (
        SEQUENCE_LENGTH,
        EXPECTED_CONFIG["sequence_length"],
    ),
    "Hidden Size": (
        HIDDEN_SIZE,
        EXPECTED_CONFIG["hidden_size"],
    ),
    "Transformer Layers": (
        NUM_LAYERS,
        EXPECTED_CONFIG["num_layers"],
    ),
    "Attention Heads": (
        NUM_HEADS,
        EXPECTED_CONFIG["num_heads"],
    ),
    "Intermediate Size": (
        INTERMEDIATE_SIZE,
        EXPECTED_CONFIG["intermediate_size"],
    ),
}


configuration_passed = True

for name, (actual, expected) in checks.items():

    if actual is None:
        print(
            f"{name:<25}: ⚠️ UNKNOWN "
            f"(expected {expected})"
        )

        configuration_passed = False

    elif actual == expected:
        print(
            f"{name:<25}: ✅ {actual}"
        )

    else:
        print(
            f"{name:<25}: ❌ {actual} "
            f"(expected {expected})"
        )

        configuration_passed = False


print("=" * 75)

if configuration_passed:
    print(
        "Configuration Check : ✅ PASSED"
    )
else:
    print(
        "Configuration Check : ⚠️ REVIEW REQUIRED"
    )

Model Configuration
Vocabulary Size      : 32,000
Sequence Length      : 512
Hidden Size          : 768
Transformer Layers   : 12
Attention Heads      : 12
Intermediate Size    : 3072
Sequence Length : ✅ 512 (matches training configuration)

Expected Training Configuration Check
Vocabulary Size          : ✅ 32000
Sequence Length          : ✅ 512
Hidden Size              : ✅ 768
Transformer Layers       : ✅ 12
Attention Heads          : ✅ 12
Intermediate Size        : ✅ 3072
Configuration Check : ✅ PASSED


In [23]:
# ============================================================
# Cell 5 - Load Tokenizer
# ============================================================

print("=" * 75)
print("Loading Tokenizer")
print("=" * 75)

tokenizer = MyGPTTokenizer.load(
    TOKENIZER_PATH
)

TOKENIZER_VOCAB_SIZE = tokenizer.vocabulary_size

print("Tokenizer Path      :", TOKENIZER_PATH)
print("Tokenizer Vocabulary:", f"{TOKENIZER_VOCAB_SIZE:,}")

if TOKENIZER_VOCAB_SIZE != VOCAB_SIZE:
    raise RuntimeError(
        "Vocabulary mismatch!\n"
        f"Tokenizer vocabulary : {TOKENIZER_VOCAB_SIZE}\n"
        f"Model vocabulary     : {VOCAB_SIZE}"
    )

print()
print("Tokenizer            : ✅ LOADED")
print("Vocabulary Match     : ✅ PASSED")

print("=" * 75)

Loading Tokenizer
Tokenizer Path      : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json
Tokenizer Vocabulary: 32,000

Tokenizer            : ✅ LOADED
Vocabulary Match     : ✅ PASSED


In [24]:
# ============================================================
# Cell 6 - Load Model + Checkpoint
# ============================================================

print("=" * 75)
print("Loading Model")
print("=" * 75)

model = MyGPTModel(config).to(DEVICE)

model.eval()

print("Model created        : ✅")

checkpoint = load_checkpoint(
    path=CHECKPOINT_PATH,
    model=model,
    optimizer=None,
    scheduler=None,
    device=DEVICE,
    restore_rng=False,
)

print()
print("Checkpoint version   :", checkpoint.get("checkpoint_version"))
print("Saved epoch          :", checkpoint.get("epoch"))
print("Saved global step    :", checkpoint.get("global_step"))
print("Saved training loss  :", checkpoint.get("train_loss"))
print("Saved validation loss:", checkpoint.get("val_loss"))

print()
print("Model checkpoint     : ✅ LOADED")

print("=" * 75)

Loading Model
Model created        : ✅

Checkpoint version   : 1.2
Saved epoch          : 0
Saved global step    : 10000
Saved training loss  : 1.3898952007293701
Saved validation loss: None

Model checkpoint     : ✅ LOADED


In [25]:
# ============================================================
# Cell 7 - Parameter Verification
# ============================================================

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("=" * 75)
print("Model Parameters")
print("=" * 75)

print(
    f"Total Parameters     : {total_parameters:,}"
)

print(
    f"Total Parameters     : "
    f"{total_parameters / 1_000_000:.2f}M"
)

print(
    f"Trainable Parameters : "
    f"{trainable_parameters:,}"
)

print("=" * 75)

EXPECTED_PARAMETERS = 110_025_216

if total_parameters == EXPECTED_PARAMETERS:
    print("Parameter Count      : ✅ MATCH")
else:
    print(
        "Parameter Count      : ⚠️ DIFFERENT"
    )
    print(
        f"Expected             : {EXPECTED_PARAMETERS:,}"
    )

Model Parameters
Total Parameters     : 110,025,216
Total Parameters     : 110.03M
Trainable Parameters : 110,025,216
Parameter Count      : ✅ MATCH


In [26]:
# ============================================================
# Cell 8 - Load TinyStories Validation Dataset
# ============================================================

try:
    from datasets import load_dataset
except ImportError:
    raise ImportError(
        "The 'datasets' package is required.\n"
        "Install it with:\n"
        "pip install datasets"
    )

print("=" * 75)
print("Loading TinyStories Validation Dataset")
print("=" * 75)

VALIDATION_DATASET_NAME = "roneneldan/TinyStories"

print("Dataset :", VALIDATION_DATASET_NAME)
print("Split   : validation")
print()

validation_dataset = load_dataset(
    VALIDATION_DATASET_NAME,
    split="validation",
)

print(
    "Validation documents :",
    f"{len(validation_dataset):,}"
)

print()
print("Dataset loading      : ✅ PASSED")

print("=" * 75)

Loading TinyStories Validation Dataset
Dataset : roneneldan/TinyStories
Split   : validation



Validation documents : 21,990

Dataset loading      : ✅ PASSED


In [27]:
# ============================================================
# Cell 9 - Dataset Inspection
# ============================================================

print("=" * 75)
print("Validation Dataset Inspection")
print("=" * 75)

print("Columns:")
print(validation_dataset.column_names)

print()

if len(validation_dataset) == 0:
    raise RuntimeError(
        "Validation dataset is empty."
    )

first_sample = validation_dataset[0]

print("First sample keys:")
print(first_sample.keys())

print()

if "text" not in first_sample:
    raise RuntimeError(
        "TinyStories validation dataset does not contain "
        "the expected 'text' field."
    )

print("First sample preview:")
print(
    first_sample["text"][:500]
)

print("=" * 75)

Validation Dataset Inspection
Columns:
['text']

First sample keys:
dict_keys(['text'])

First sample preview:
Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."

After playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.


In [ ]:
# ============================================================
# Cell 10 - Tokenization Helper
# ============================================================

def encode_text(tokenizer, text):
    """
    Convert text into integer token IDs.

    Supports common MyGPTTokenizer interfaces.
    """

    if not isinstance(text, str):
        raise TypeError(
            f"Expected text to be str, got {type(text)}"
        )

    # --------------------------------------------------------
    # Preferred MyGPTTokenizer API
    # --------------------------------------------------------

    if hasattr(tokenizer, "encode"):
        result = tokenizer.encode(text)

        # Some wrappers return an object
        if hasattr(result, "ids"):
            result = result.ids

        if isinstance(result, torch.Tensor):
            result = result.detach().cpu().tolist()

        if isinstance(result, (list, tuple)):
            return [int(x) for x in result]

    # --------------------------------------------------------
    # Explicit tokenize API
    # --------------------------------------------------------

    if hasattr(tokenizer, "tokenize"):
        result = tokenizer.tokenize(text)

        if hasattr(result, "ids"):
            result = result.ids

        if isinstance(result, torch.Tensor):
            result = result.detach().cpu().tolist()

        if isinstance(result, (list, tuple)):
            return [int(x) for x in result]

    raise RuntimeError(
        "Could not determine how to encode text with "
        "MyGPTTokenizer."
    )


# Test tokenizer

test_text = validation_dataset[0]["text"]

test_tokens = encode_text(
    tokenizer,
    test_text,
)

print("=" * 75)
print("Tokenizer Test")
print("=" * 75)

print("Characters :", len(test_text))
print("Tokens     :", len(test_tokens))

print(
    "First tokens:",
    test_tokens[:20]
)

if len(test_tokens) == 0:
    raise RuntimeError(
        "Tokenizer produced zero tokens."
    )

print()
print("Tokenizer test       : ✅ PASSED")

print("=" * 75)